In [ ]:
import time
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori

# Doc du lieu: moi dong cua file la 1 giao dich, cac muc tach nhau bang dau phay
transactions = []
with open("groceries.csv", "r", encoding="utf-8") as f:
    for line in f:
        items = [item.strip() for item in line.strip().split(",") if item.strip()]
        if items:
            transactions.append(items)

# Chuyen doi sang one-hot encoding (kieu bool de Apriori xu ly hieu qua)
te = TransactionEncoder()
te_ary = te.fit(transactions).transform(transactions)
basket = pd.DataFrame(te_ary, columns=te.columns_).astype(bool)
print("Bang one-hot:", basket.shape)

# Tim tat ca tap pho bien bang Apriori, min_sup tu 0.01% den 0.1%
# max_len=2: gioi han kich thuoc tap (cap mat hang) de tranh bung no bo nho
ds_min_sup = [0.0001, 0.0002, 0.0003, 0.0004, 0.0005,
              0.0006, 0.0007, 0.0008, 0.0009, 0.0010]

ket_qua = []
for min_sup in ds_min_sup:
    start = time.time()
    itemsets = apriori(basket, min_support=min_sup, use_colnames=True, max_len=2)
    elapsed = time.time() - start
    ket_qua.append({
        "min_sup": min_sup,
        "min_sup_%": round(min_sup * 100, 2),
        "min_support_count": round(min_sup * len(basket)),
        "so_tap_pho_bien": len(itemsets),
        "thoi_gian (s)": round(elapsed, 4)
    })
    print(f"min_sup = {min_sup*100:.2f}% (count >= {min_sup*len(basket):.1f}): "
          f"{len(itemsets)} tap pho bien | {elapsed:.4f}s")

bang_apriori = pd.DataFrame(ket_qua)
print("\n--- BANG TONG HOP KET QUA APRIORI ---")
display(bang_apriori)

# Xem chi tiet tap pho bien voi nguong nho nhat (0.01%)
itemsets_min = apriori(basket, min_support=0.0001, use_colnames=True, max_len=2)
itemsets_min["do_dai"] = itemsets_min["itemsets"].apply(len)

print("\n--- So tap pho bien theo do dai (min_sup = 0.01%) ---")
display(itemsets_min["do_dai"].value_counts().sort_index().rename("so_luong"))

print("\n--- 10 tap pho bien co support cao nhat (min_sup = 0.01%) ---")
top = itemsets_min.sort_values("support", ascending=False).head(10).copy()
top["support"] = top["support"].round(4)
top["itemsets"] = top["itemsets"].apply(lambda s: ", ".join(sorted(s)))
display(top[["support", "itemsets"]])

print("\n--- 10 cap mat hang mua kem voi support thap (min_sup = 0.01%) ---")
hien = itemsets_min[itemsets_min["do_dai"] == 2].head(10).copy()
hien["support"] = hien["support"].round(4)
hien["itemsets"] = hien["itemsets"].apply(lambda s: ", ".join(sorted(s)))
display(hien[["support", "itemsets"]])